# Dataset arithmetic with `sans_fitter.data.ops`

This notebook shows how to combine SANS datasets with arithmetic operations,
similar to SasView's *Data Operation* utility, and fit the result:

- `subtract(sample, background)` - empty-cell / solvent subtraction
- `multiply(data, k)` / `divide(data, k)` - rescaling (absolute units, transmission)
- `subtract(data, c)` - flat background subtraction
- `add(a, b)` - combining datasets

All functions return a new, **fit-ready** `Data1D` with propagated uncertainties
(`dy = sqrt(dy_a² + dy_b²)` for addition/subtraction). The result is injected into a
fitter with `SANSFitter.set_data()`.

In [ ]:
import numpy as np
import plotly.graph_objects as go

from sans_fitter import SANSFitter, data_ops

## 1. Simulate a sample and a background measurement

We simulate two "measurements" on the same Q grid and save them as CSV files:

- **sample**: sphere scattering (radius 60 Å) on top of a flat instrument background
- **background**: the flat background alone

Each gets 3% counting noise.

In [ ]:
from sasmodels.core import load_model
from sasmodels.data import empty_data1D
from sasmodels.direct_model import DirectModel

rng = np.random.default_rng(seed=42)
q = np.logspace(np.log10(0.008), np.log10(0.35), 80)

kernel = load_model('sphere', dtype='single', platform='dll')
calculator = DirectModel(empty_data1D(q), kernel)
i_sphere = calculator(radius=60.0, scale=0.005, background=0.0, sld=4.0, sld_solvent=1.0)

BACKGROUND_LEVEL = 0.08   # flat instrument/solvent background
TRANSMISSION = 0.8        # sample transmission factor

i_sample = i_sphere + BACKGROUND_LEVEL
di_sample = 0.03 * i_sample
i_sample = i_sample + rng.normal(0.0, di_sample)

i_bkg = np.full_like(q, BACKGROUND_LEVEL)
di_bkg = 0.03 * i_bkg
i_bkg = i_bkg + rng.normal(0.0, di_bkg)

for filename, intensity, error in [
    ('example_sample.csv', i_sample, di_sample),
    ('example_background.csv', i_bkg, di_bkg),
]:
    with open(filename, 'w') as f:
        f.write('Q,I,dI\n')
        for row in zip(q, intensity, error):
            f.write(f'{row[0]},{row[1]},{row[2]}\n')
    print(f'wrote {filename}')

## 2. Load the datasets

`data_ops.load()` is the standalone equivalent of `SANSFitter.load_data()`: it returns a
fit-ready `Data1D` instead of storing it on a fitter, so you can manipulate it first.

In [ ]:
sample = data_ops.load('example_sample.csv')
background = data_ops.load('example_background.csv')

print(f'sample:     {len(sample.x)} points, Q = [{sample.x.min():.4g}, {sample.x.max():.4g}]')
print(f'background: {len(background.x)} points, Q = [{background.x.min():.4g}, {background.x.max():.4g}]')

## 3. Subtract the background and correct for transmission

`subtract(a, b)` returns `a − b` (order matters). Uncertainties are combined in
quadrature.
Scalar operations (`divide(net, 0.8)`) scale `y` and `dy` and d not modify the Q grid.

In [ ]:
net = data_ops.subtract(sample, background)
net = data_ops.divide(net, TRANSMISSION)

print(f'title: {net.title}')
for p in net.process:
    print(f'process: {p.description}')

# error propagation: dy_net = sqrt(dy_sample² + dy_bkg²) / transmission
expected = np.sqrt(sample.dy**2 + background.dy**2) / TRANSMISSION
print('dy propagated correctly:', np.allclose(net.dy, expected))

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=sample.x, y=sample.y, mode='markers', name='sample',
                         error_y={'type': 'data', 'array': sample.dy, 'visible': True}))
fig.add_trace(go.Scatter(x=background.x, y=background.y, mode='markers', name='background',
                         error_y={'type': 'data', 'array': background.dy, 'visible': True}))
fig.add_trace(go.Scatter(x=net.x, y=net.y, mode='markers', name='(sample − background) / T',
                         error_y={'type': 'data', 'array': net.dy, 'visible': True}))
fig.update_layout(xaxis={'type': 'log', 'title': 'Q (Å⁻¹)'},
                  yaxis={'type': 'log', 'title': 'I(Q) (cm⁻¹)'},
                  title='Background subtraction', template='plotly_white')
fig.show()

## 4. Fit the corrected dataset

`SANSFitter.set_data()` accepts any in-memory `Data1D` (arithmetic results,
simulated data, etc.) validates it and makes it fit-ready.

In [ ]:
fitter = SANSFitter()
fitter.set_data(net)
fitter.set_model('sphere')

fitter.set_param('radius', value=40.0, min=10.0, max=150.0, vary=True)
fitter.set_param('scale', value=0.001, min=1e-4, max=1.0, vary=True)
fitter.set_param('background', value=0.001, min=0.0, max=1.0, vary=True)
fitter.set_param('sld', value=4.0, vary=False)
fitter.set_param('sld_solvent', value=1.0, vary=False)

result = fitter.fit(engine='bumps', method='amoeba')

print('\nGround truth vs fit:')
print(f'  radius: 60.0 Å  -> fitted {result["parameters"]["radius"]["value"]:.1f} Å')
print(f'  scale:  {0.005 / TRANSMISSION:.5f} -> fitted {result["parameters"]["scale"]["value"]:.5f}')

In [ ]:
fitter.plot_results(show_residuals=True, log_scale=True)

## 5. Things to know

**Mismatched Q grids are rejected.** Both datasets must share the same Q grid
(x-values matching within 1%). Interpolation onto a common grid is not yet supported.
You need to rebin first. The error names both datasets and their Q ranges:

In [ ]:
from sasdata.dataloader.data_info import Data1D

other_grid = Data1D(x=q * 1.5, y=np.ones_like(q), dy=np.full_like(q, 0.1))
try:
    data_ops.subtract(sample, other_grid)
except ValueError as e:
    print(f'ValueError: {e}')

**Other caveats:**

- **Missing uncertainties (dI)** on an operand trigger a warning - they are treated as
  zero in error propagation. Error-free results warn again at fit time (the `lmfit`
  engine falls back to unit weights; `bumps` refuses to fit).
- **NaN points** propagate through the arithmetic; they are masked in the result
  (excluded from fits) and a warning reports the masked count.
- **Resolution (dQ) propagation** through arithmetic is *not validated* upstream -
  a warning is emitted when any operand carries `dx`/`dxl`/`dxw`. Treat resolution on
  results with care, especially for slit-smeared data.
- Every operation appends a `Process` entry to the result, so the provenance survives
  in saved CanSAS output.

In [ ]:
fitter.save_results('background_subtracted_fit.csv')